In [5]:
# --- Célula: Data Masking ---
from pyspark.sql import SparkSession
from pyspark.sql.functions import udf, col
from pyspark.sql.types import StringType

# Inicializar SparkSession se ainda não estiver inicializada
spark = SparkSession.builder.appName("DataMasking").getOrCreate()

# Função de mascaramento de CPF
def mask_cpf(cpf):
  if cpf and len(cpf) == 11:
    return f"***.***.{cpf[-6:-3]}-{cpf[-2:]}"
  return cpf

# Função de mascaramento de e-mail
def mask_email(email):
  if email and "@" in email:
    local, domain = email.split("@")
    return f"{local[0]}***@{domain[0]}***"
  return email

# Registrar UDFs
mask_cpf_udf = udf(mask_cpf, StringType())
mask_email_udf = udf(mask_email, StringType())

# Dados sensíveis
df_clientes = spark.createDataFrame([
("12345678901", "joao@example.com"),
("98765432100", "maria@example.com")
], ["cpf", "email"])

# Aplicar mascaramento
df_masked = df_clientes.withColumn("cpf_masked", mask_cpf_udf(col("cpf"))) \
.withColumn("email_masked", mask_email_udf(col("email")))

print("Dados Originais vs Mascarados:")
df_masked.show(truncate=False)

Dados Originais vs Mascarados:
+-----------+-----------------+--------------+------------+
|cpf        |email            |cpf_masked    |email_masked|
+-----------+-----------------+--------------+------------+
|12345678901|joao@example.com |***.***.678-01|j***@e***   |
|98765432100|maria@example.com|***.***.432-00|m***@e***   |
+-----------+-----------------+--------------+------------+



### Explicação do Mascaramento de Dados

Nesta implementação, utilizamos **PySpark UDFs (User Defined Functions)** para transformar dados sensíveis em versões mascaradas:

1.  **Mascaramento de CPF**:
    * Mantém os últimos 6 dígitos visíveis no formato `***.***.XXX-XXX`.
    * Utiliza f-strings e fatiamento de strings (`cpf[-6:-3]`) para capturar as partes finais.

2.  **Mascaramento de E-mail**:
    * Preserva apenas a primeira letra do nome de usuário e a primeira letra do domínio.
    * O formato resultante é `u***@d***`.

Essas funções são registradas como UDFs e aplicadas às colunas do DataFrame original usando o método `.withColumn()`.